In [1]:
import pathlib

import numpy as np
import pandas as pd

In [2]:
from trait_prediction.utils import (
    read_phenotype_data,
    read_interpro_features,
    read_rast_features,
)

In [3]:
data_folder = pathlib.Path("../data/raw/")

metabolic_phenotype_file = data_folder / "metabolic_phenotypes_bacdive.tsv"
non_metabolic_phenotype_file = data_folder / "non_metabolic_phenotypes_bacdive.tsv"
rast_feature_file = data_folder / "rast_features.tsv"
interpro_feature_file = data_folder / "interpro_features.tsv"

# Read phenotype data into Phenotype objects

In [4]:
from trait_prediction.main import Phenotype
print(Phenotype.__doc__)


    Class that represents a phenotype.

        Parameters
        ---------
        raw_phenotype_data : pd.Series
            Pandas Series containing the raw phenotype data.
        name : str
            Name of the phenotype.
        category : str
            Category of the phenotype.

        Attributes
        ---------
        phenotype_data : pd.Series
            Pandas Series containing the filtered phenotype data.
        name : str
            Name of the phenotype.
        category : str
            Category of the phenotype.
        feature_data : pd.DataFrame
            Pandas DataFrame containing the feature data.
        feature_type : Union[str, None]
            Type of the feature data.
    


In [5]:
rast_metabolic_phenotypes: list[Phenotype] = read_phenotype_data(metabolic_phenotype_file)
rast_non_metabolic_phenotypes = read_phenotype_data(non_metabolic_phenotype_file)

interpro_metabolic_phenotypes = read_phenotype_data(metabolic_phenotype_file)
interpro_non_metabolic_phenotypes = read_phenotype_data(non_metabolic_phenotype_file)

There are two unknown metabolic phenotypes

In [6]:
[m for m in rast_metabolic_phenotypes if m.category.startswith("unknown")]

[Phenotype (name=unnamed_13, category=unknown, size=0),
 Phenotype (name=ncbi_tax_id-subspecies, category=unknown, size=97)]

# Read features data into new format

In [7]:
rast_features, rast_sso_dict = read_rast_features(rast_feature_file)
interpro_features, interpro_ipr_dict = read_interpro_features(interpro_feature_file)

In [8]:
rast_features

,SSO:000020125,SSO:000033083,SSO:000005939,SSO:000004884,SSO:000020349,SSO:000043478,SSO:000000080,SSO:000007884,SSO:000017709,SSO:000041924,...,SSO:000007881,SSO:000021617,SSO:000009584,SSO:000012182,SSO:000021803,SSO:000017551,SSO:000002621,SSO:000039031,SSO:000032815,SSO:000001251
GenomeID,,,,,,,,,,,,,,,,,,,,,
GCF_005938105,0,0,0,0,0,0,0,0,0,0,...,0,0,1,1,0,0,0,0,0,0
GCF_000284515,0,0,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
GCF_001708125,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCF_002224365,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCF_001746755,0,0,1,0,0,0,0,0,0,0,...,0,0,1,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GCF_016863215,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCF_002899475,1,0,1,0,0,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,1
GCF_000242075,0,0,0,0,0,0,0,1,0,0,...,0,0,1,1,0,0,0,0,0,0


In [9]:
interpro_features

,IPR016372,IPR023894,IPR022765,IPR024421,IPR045517,IPR012489,IPR024077,IPR026410,IPR018592,IPR007646,...,IPR009513,IPR015942,IPR014330,IPR043931,IPR040572,IPR015170,IPR014967,IPR018758,IPR027826,IPR015256
GenomeID,,,,,,,,,,,,,,,,,,,,,
GCF_004362145,0,0,0,0,0,0,1,0,0,0,...,0,1,0,0,0,0,0,0,0,0
GCF_000427095,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,0,0,0,0,0,0
GCF_009711225,0,0,0,0,1,0,1,0,0,0,...,0,1,0,0,0,0,0,0,0,0
GCF_900101385,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,0,0,0,0,0,0
GCF_900102145,0,0,1,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GCF_900091575,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCF_003265355,0,0,0,0,0,0,1,0,0,0,...,0,1,0,0,0,0,0,0,0,0
GCF_000613725,0,0,0,0,0,0,1,0,0,0,...,0,1,0,0,0,0,0,0,0,0


There are no RAST features that are annotated incorrectly

In [10]:
[c for c in rast_features.columns if not c.startswith('SSO')]

[]

There is one Interpro feature that is annotated incorrectly (without IPR)

In [11]:
[c for c in interpro_features.columns if not c.startswith('IPR')]

['-']

## Attaching the feature data to the Phenotype class

Let us look at an example where we attach the rast_features to the "glucose--carbon_source" phenotype

In [12]:
glucose = [p for p in rast_metabolic_phenotypes if p.name == "glucose" and p.category == "carbon_source"][0]
glucose

Phenotype (name=glucose, category=carbon_source, size=1606)

In [13]:
glucose.set_feature_data(rast_features, "rast")
glucose

Phenotype (name=glucose, category=carbon_source, size=1606)

In [14]:
# NOTE: This returns a new dataframe
glucose.feature_data

,SSO:000020125,SSO:000033083,SSO:000005939,SSO:000004884,SSO:000020349,SSO:000043478,SSO:000000080,SSO:000007884,SSO:000017709,SSO:000041924,...,SSO:000007881,SSO:000021617,SSO:000009584,SSO:000012182,SSO:000021803,SSO:000017551,SSO:000002621,SSO:000039031,SSO:000032815,SSO:000001251
GenomeID,,,,,,,,,,,,,,,,,,,,,
GCF_000010785,0,0,1,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
GCF_000014145,0,0,0,0,0,0,0,0,0,0,...,0,0,1,1,0,0,0,0,0,0
GCF_000014885,0,0,1,0,0,0,0,0,0,0,...,0,0,1,1,0,0,0,0,0,0
GCF_000015285,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
GCF_000016785,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GCF_902167755,0,0,1,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
GCF_902833725,0,0,1,0,0,0,0,0,0,0,...,0,0,1,1,0,0,0,0,0,0
GCF_902859775,0,0,1,0,0,0,0,0,0,0,...,0,0,1,1,0,0,0,0,0,0


In [15]:
low_var_features, correlated_features_dict = glucose.filter_feature_data(
    variance_threshold=0.05, correlation_treshold=0.95)

In [16]:
fd = glucose.feature_data
fd

,SSO:000005939,SSO:000000080,SSO:000002624,SSO:000000292,SSO:000036433,SSO:000000705,SSO:000011933,SSO:000024328,SSO:000035324,SSO:000002475,...,SSO:000036573,SSO:000002117,SSO:000000074,SSO:000019142,SSO:000000027,SSO:000006963,SSO:000004390,SSO:000024474,SSO:000009584,SSO:000012182
GenomeID,,,,,,,,,,,,,,,,,,,,,
GCF_000010785,1,0,0,0,0,0,0,0,0,1,...,1,0,0,0,0,0,0,1,0,1
GCF_000014145,0,0,1,1,0,0,0,1,0,1,...,1,0,1,0,0,0,0,1,1,1
GCF_000014885,1,0,1,0,1,0,0,0,0,0,...,1,1,1,0,0,1,1,1,1,1
GCF_000015285,0,0,1,0,1,0,0,0,0,0,...,1,1,1,0,1,0,1,1,1,0
GCF_000016785,0,0,1,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GCF_902167755,1,0,1,0,0,1,1,0,0,0,...,1,0,0,0,0,0,0,0,0,1
GCF_902833725,1,0,1,0,1,0,0,0,1,0,...,1,1,1,0,1,0,1,1,1,1
GCF_902859775,1,0,1,0,1,0,0,1,1,0,...,1,1,1,0,1,0,1,1,1,1


## Saving the file

In [17]:
output_folder = pathlib.Path(f"../data/processed/{glucose.category}")
output_folder.mkdir(parents=True, exist_ok=True)
output_file = output_folder / f"{glucose.name}.pkl"
glucose.save(str(output_file))